# clustering_v2 — Team 1 behavioral segments

**Question:** What natural behavior groups exist among developers, within each lifecycle band?

**Method:** `dev_profile_final_v4` → median impute + `RobustScaler` → **SVD(50)** → **HDBSCAN** on scaled features. UMAP is **visualization only**.

**Strata:** `active`, `cooling`, `dormant` (separate cluster fits per stratum).

**Prerequisite:** [FeatureEngineering_v3.ipynb](FeatureEngineering_v3.ipynb) through `dev_profile_final_v4` in `developer_project.duckdb`.

**Docs:** [docs/clustering_breakdown.md](docs/clustering_breakdown.md) · **Archive:** [clustering_v1.ipynb](clustering_v1.ipynb).


In [2]:
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import duckdb
import hdbscan
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import umap
from IPython.display import display
from sklearn.decomposition import TruncatedSVD
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")

DB_PATH = "developer_project.duckdb"
OUTPUT_DIR = Path("outputs/clustering/v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_N = 100_000
RANDOM_STATE = 42
HASH_SALT = 42
SVD_COMPONENTS = 50
MIN_ROWS = 200

UMAP_PARAMS = dict(
    n_components=2,
    n_neighbors=30,
    min_dist=0.1,
    metric="euclidean",
    init="random",
    random_state=RANDOM_STATE,
)

STRATA_ORDER = ["active", "cooling", "dormant"]

con = duckdb.connect(DB_PATH)
print("Connected:", DB_PATH)
print("Output:", OUTPUT_DIR.resolve())


Connected: developer_project.duckdb
Output: /Users/pmazolew/Documents/GitHub/Spring2026_IndustryProject/outputs/clustering/v2


## 1. Shared helpers


In [3]:
EXCLUDE_EXACT = {
    "developer_id",
    "contact_created_date", "contact_first_activity_date", "contact_last_activity_date",
    "account_id", "account_type", "country", "region", "industry_segment_vertical",
    "program_application_source", "organization_english_name", "normalized_account_name",
    "wwfo_category", "wwfo_target_list",
    "behavior_journey_stage_30d", "behavior_journey_rank_30d",
    "current_journey_state_30d", "current_journey_rank_30d",
    "final_lifecycle_status", "persona", "persona_confidence_tier", "developer_effort_level",
    "dormancy_status", "user_type", "max_stage_reached",
    "last_meaningful_week_start", "lifetime_first_activity_date", "lifetime_last_activity_date",
    "missing_contact_metadata_flag",
}
RAW_LIFETIME_SUPERSEDED = {
    "lifetime_activity_count", "lifetime_activity_score_sum", "lifetime_build_count",
    "lifetime_high_effort_count", "lifetime_total_confidence_weighted_effort",
    "lifetime_effort_x_activity_score_sum",
}

PROFILE_COLS = [
    "activity_count_0_30d", "log_activity_count_0_30d", "log_activity_count_30_90d",
    "build_count_0_30d", "log_build_count_0_30d", "log_build_count_30_90d",
    "lifetime_learn_count", "lifetime_build_count", "lifetime_champion_count",
    "lifetime_discover_count", "developer_effort_score", "days_since_last_activity",
    "cuda_share", "genai_share", "robotics_share", "simulation_share",
]


def should_exclude(col: str) -> bool:
    if col in EXCLUDE_EXACT or col in RAW_LIFETIME_SUPERSEDED:
        return True
    return col.startswith("first_activity_date") or col.startswith("last_activity_date")


STRATA = {
    "active": {
        "label": "Active (recent 30d activity)",
        "where_sql": (
            "is_activated = 1 AND COALESCE(dormancy_status, '') = 'Active' "
            "AND COALESCE(has_activity_0_30d, 0) = 1"
        ),
        "stratify_col": "persona",
        "min_cluster_frac": 0.005,
        "min_samples": 15,
    },
    "cooling": {
        "label": "Cooling",
        "where_sql": "COALESCE(dormancy_status, '') = 'Cooling'",
        "stratify_col": "persona",
        "min_cluster_frac": 0.007,
        "min_samples": 20,
    },
    "dormant": {
        "label": "Dormant",
        "where_sql": "COALESCE(dormancy_status, '') = 'Dormant'",
        "stratify_col": "persona",
        "min_cluster_frac": 0.007,
        "min_samples": 25,
    },
}


@dataclass
class StratumRunResult:
    stratum_key: str
    cohort_n: int
    sample_n: int
    n_features: int
    n_clusters: int
    noise_pct: float
    min_cluster_size: int
    output_dir: Path
    skipped: bool = False
    skip_reason: str = ""


def _sample_stratum(con, stratum_key, where_sql, stratify_col, sample_target, hash_salt):
    view = f"clustering_cohort_v2_{stratum_key}"
    sample_table = f"clustering_sample_v2_{stratum_key}"
    con.execute(f"CREATE OR REPLACE VIEW {view} AS SELECT * FROM dev_profile_final_v4 WHERE {where_sql}")
    cohort_n = con.execute(f"SELECT COUNT(*) FROM {view}").fetchone()[0]
    target = min(sample_target, cohort_n)
    if cohort_n == 0:
        return 0, 0, sample_table
    if cohort_n <= target:
        con.execute(f"CREATE OR REPLACE TABLE {sample_table} AS SELECT * FROM {view}")
        return cohort_n, cohort_n, sample_table
    con.execute(f"""
        CREATE OR REPLACE TABLE {sample_table} AS
        WITH cohort AS (SELECT * FROM {view}),
        targets AS (
            SELECT {stratify_col}, COUNT(*) AS stratum_n,
                GREATEST(1, CAST(ROUND(COUNT(*) * 1.0 / SUM(COUNT(*)) OVER () * {target}) AS BIGINT)) AS target_n
            FROM cohort GROUP BY {stratify_col}
        ),
        ranked AS (
            SELECT c.*,
                ROW_NUMBER() OVER (PARTITION BY c.{stratify_col} ORDER BY hash(c.developer_id || '{hash_salt}')) AS rn,
                t.target_n
            FROM cohort c JOIN targets t USING ({stratify_col})
        )
        SELECT * EXCLUDE (rn, target_n) FROM ranked WHERE rn <= target_n
    """)
    sample_n = con.execute(f"SELECT COUNT(*) FROM {sample_table}").fetchone()[0]
    if sample_n > target:
        con.execute(f"""
            CREATE OR REPLACE TABLE {sample_table} AS
            SELECT * EXCLUDE (rn) FROM (
                SELECT *, ROW_NUMBER() OVER (ORDER BY hash(developer_id || '{hash_salt}')) AS rn
                FROM {sample_table}
            ) WHERE rn <= {target}
        """)
        sample_n = con.execute(f"SELECT COUNT(*) FROM {sample_table}").fetchone()[0]
    return cohort_n, sample_n, sample_table


def _build_matrices(con, sample_table):
    profile_cols = [r[0] for r in con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_name = 'dev_profile_final_v4' ORDER BY ordinal_position"
    ).fetchall()]
    sample_cols = [r[0] for r in con.execute(
        f"SELECT column_name FROM information_schema.columns "
        f"WHERE table_name = '{sample_table}' ORDER BY ordinal_position"
    ).fetchall()]
    candidate = [c for c in profile_cols if c in sample_cols and not should_exclude(c)]
    df = con.execute(f"SELECT * FROM {sample_table}").fetchdf()
    numeric_cols = [c for c in candidate if pd.api.types.is_numeric_dtype(df[c])]
    preprocessor = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ])
    X_scaled = preprocessor.fit_transform(df[numeric_cols])
    keep = np.var(X_scaled, axis=0) > 1e-12
    feature_names = [numeric_cols[i] for i, k in enumerate(keep) if k]
    X_scaled = X_scaled[:, keep]
    n_svd = min(SVD_COMPONENTS, X_scaled.shape[1] - 1, X_scaled.shape[0] - 1)
    svd = TruncatedSVD(n_components=n_svd, random_state=RANDOM_STATE)
    X_svd = svd.fit_transform(X_scaled)
    return df, X_scaled, X_svd, feature_names, preprocessor, svd


def _behavioral_segment_name(g, stratum_key):
    def m(col):
        return float(g[col].mean()) if col in g.columns and g[col].notna().any() else 0.0

    learn = m("lifetime_learn_count") + m("log_activity_count_0_30d")
    build = m("lifetime_build_count") + m("build_count_0_30d") + m("log_build_count_0_30d")
    champ = m("lifetime_champion_count")
    discover = m("lifetime_discover_count")
    recent = m("activity_count_0_30d")

    if build >= max(learn, champ, discover, 1e-9) and build > 0:
        base = "Builders"
    elif champ >= max(learn, build, discover, 1e-9) and champ > 0:
        base = "Champions"
    elif learn >= max(build, champ, discover, 1e-9) and learn > 0:
        base = "Learners"
    elif discover >= max(learn, build, champ, 1e-9) and discover > 0:
        base = "Explorers"
    elif recent > 0:
        base = "Active_Mixed"
    else:
        base = "Low_Signal"

    if stratum_key == "dormant" and build > learn and build > 0:
        return f"Dormant_Former_{base}"
    if stratum_key == "cooling":
        return f"Cooling_{base}"
    return base


def _segment_description(row, stratum_key):
    if row.get("noise_flag"):
        return "Sparse or heterogeneous; not assigned to a dense HDBSCAN cluster."
    name = row.get("behavioral_segment_name", "Segment")
    persona = row.get("top_persona", "")
    journey = row.get("top_journey_state_30d", "")
    n = row.get("n", 0)
    pct = row.get("pct_of_sample", 0)
    return (
        f"{name} ({stratum_key}): n={n:,} ({pct}% of sample). "
        f"Dominant persona {persona}. Typical journey {journey}."
    )


def build_cluster_summary(df, labels, stratum_key, sample_n):
    tmp = df.copy()
    tmp["hdbscan_cluster"] = labels
    profile_cols = [c for c in PROFILE_COLS if c in tmp.columns]
    rows = []
    for cid, g in tmp.groupby("hdbscan_cluster", observed=True):
        row = {
            "dormancy_segment": stratum_key,
            "hdbscan_cluster": int(cid),
            "n": len(g),
            "pct_of_sample": round(100.0 * len(g) / sample_n, 2),
            "noise_flag": int(cid == -1),
        }
        if cid >= 0 and len(g):
            persona_counts = g["persona"].value_counts()
            row["top_persona"] = persona_counts.index[0]
            row["top_persona_pct"] = round(100.0 * persona_counts.iloc[0] / len(g), 2)
            if "current_journey_state_30d" in g.columns and g["current_journey_state_30d"].notna().any():
                row["top_journey_state_30d"] = g["current_journey_state_30d"].mode().iloc[0]
            if "final_lifecycle_status" in g.columns and g["final_lifecycle_status"].notna().any():
                row["top_lifecycle_status"] = g["final_lifecycle_status"].mode().iloc[0]
            if "dormancy_status" in g.columns:
                row["top_dormancy_status"] = g["dormancy_status"].mode().iloc[0]
            row["behavioral_segment_name"] = _behavioral_segment_name(g, stratum_key)
        else:
            row["top_persona"] = ""
            row["top_persona_pct"] = 0.0
            row["top_journey_state_30d"] = ""
            row["top_lifecycle_status"] = ""
            row["top_dormancy_status"] = ""
            row["behavioral_segment_name"] = "Unclustered_Noise"
        for c in profile_cols:
            val = g[c].mean()
            row[f"mean_{c}"] = None if pd.isna(val) else float(val)
        row["segment_description"] = _segment_description(row, stratum_key)
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["noise_flag", "n"], ascending=[True, False])


def run_stratum(con, stratum_key, fit_umap=False):
    cfg = STRATA[stratum_key]
    out_dir = OUTPUT_DIR / stratum_key
    out_dir.mkdir(parents=True, exist_ok=True)

    cohort_n, sample_n, sample_table = _sample_stratum(
        con, stratum_key, cfg["where_sql"], cfg["stratify_col"], SAMPLE_N, HASH_SALT
    )
    if cohort_n == 0:
        return StratumRunResult(stratum_key, 0, 0, 0, 0, 0.0, 0, out_dir, True, "empty cohort")
    if sample_n < MIN_ROWS:
        return StratumRunResult(stratum_key, cohort_n, sample_n, 0, 0, 0.0, 0, out_dir, True, f"n < {MIN_ROWS}")

    df, X_scaled, X_svd, feature_names, preprocessor, svd = _build_matrices(con, sample_table)
    mcs = max(30, int(sample_n * cfg["min_cluster_frac"]))
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=mcs,
        min_samples=cfg["min_samples"],
        metric="euclidean",
        cluster_selection_method="eom",
    )
    labels = clusterer.fit_predict(X_svd)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_pct = round(100.0 * (labels == -1).sum() / len(labels), 2)

    results = pd.DataFrame({
        "developer_id": df["developer_id"].values,
        "dormancy_segment": stratum_key,
        "hdbscan_cluster": labels,
        "hdbscan_probability": getattr(clusterer, "probabilities_", np.ones(len(labels))),
        "noise_flag": (labels == -1).astype(int),
        "persona": df["persona"].values,
        "dormancy_status": df["dormancy_status"].values,
        "current_journey_state_30d": df["current_journey_state_30d"].values,
        "final_lifecycle_status": df["final_lifecycle_status"].values,
        "behavior_journey_stage_30d": df["behavior_journey_stage_30d"].values,
    })

    if fit_umap:
        emb = umap.UMAP(**UMAP_PARAMS).fit_transform(X_scaled)
        results["umap_x"] = emb[:, 0]
        results["umap_y"] = emb[:, 1]

    summary = build_cluster_summary(df, labels, stratum_key, sample_n)
    results.to_parquet(out_dir / "cluster_results.parquet", index=False)
    summary.to_csv(out_dir / "cluster_summary_table.csv", index=False)
    joblib.dump({"preprocessor": preprocessor, "svd": svd, "feature_names": feature_names}, out_dir / "preprocessor.joblib")

    con.register(f"clusters_v2_{stratum_key}", results)
    con.execute(f"DROP TABLE IF EXISTS developer_clusters_v2_{stratum_key}")
    con.execute(f"CREATE TABLE developer_clusters_v2_{stratum_key} AS SELECT * FROM clusters_v2_{stratum_key}")
    con.unregister(f"clusters_v2_{stratum_key}")

    return StratumRunResult(stratum_key, cohort_n, sample_n, X_scaled.shape[1], n_clusters, noise_pct, mcs, out_dir)

print("Helpers loaded.")


Helpers loaded.


## 2. Run clustering (SVD + HDBSCAN per stratum)


In [4]:
run_results = []
result_frames = []

for key in STRATA_ORDER:
    print(f"\n{'=' * 60}\nStratum: {key} — {STRATA[key]['label']}\n{'=' * 60}")
    res = run_stratum(con, key, fit_umap=False)
    run_results.append({k: v for k, v in res.__dict__.items() if k != "output_dir"})
    print(
        f"  cohort_n={res.cohort_n:,}  sample_n={res.sample_n:,}  "
        f"features={res.n_features}  clusters={res.n_clusters}  noise={res.noise_pct}%"
        + (f"  SKIPPED: {res.skip_reason}" if res.skipped else "")
    )
    if not res.skipped:
        result_frames.append(pd.read_parquet(res.output_dir / "cluster_results.parquet"))

run_summary_df = pd.DataFrame(run_results)
run_summary_df.to_csv(OUTPUT_DIR / "run_summary.csv", index=False)
display(run_summary_df)



Stratum: active — Active (recent 30d activity)
  cohort_n=418,049  sample_n=99,999  features=101  clusters=58  noise=25.4%

Stratum: cooling — Cooling


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['activity_per_active_day_0_30d' 'build_share_0_30d'
 'high_effort_share_0_30d' 'confidence_weighted_effort_per_activity_0_30d'
 'score_effort_misalignment_share_0_30d' 'days_since_last_activity_0_30d']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


  cohort_n=356,500  sample_n=100,000  features=73  clusters=38  noise=30.23%

Stratum: dormant — Dormant


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['activity_per_active_day_0_30d' 'build_share_0_30d'
 'high_effort_share_0_30d' 'confidence_weighted_effort_per_activity_0_30d'
 'score_effort_misalignment_share_0_30d' 'days_since_last_activity_0_30d'
 'activity_velocity_0_30_vs_30_90' 'build_velocity_0_30_vs_30_90']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


  cohort_n=5,304,852  sample_n=100,000  features=55  clusters=42  noise=41.49%


,stratum_key,cohort_n,sample_n,n_features,n_clusters,noise_pct,min_cluster_size,skipped,skip_reason
0,active,418049,99999,101,58,25.40,499,False,
1,cooling,356500,100000,73,38,30.23,700,False,
2,dormant,5304852,100000,55,42,41.49,700,False,


## 3. Combine and persist


In [5]:
if result_frames:
    combined_results = pd.concat(result_frames, ignore_index=True)
    combined_results.to_parquet(OUTPUT_DIR / "cluster_results_all.parquet", index=False)

    summary_parts = [
        pd.read_csv(OUTPUT_DIR / k / "cluster_summary_table.csv")
        for k in STRATA_ORDER
        if (OUTPUT_DIR / k / "cluster_summary_table.csv").exists()
    ]
    summary_all = pd.concat(summary_parts, ignore_index=True)
    summary_all.to_csv(OUTPUT_DIR / "cluster_summary_table_all.csv", index=False)

    con.register("developer_clusters_v2_df", combined_results)
    con.execute("DROP TABLE IF EXISTS developer_clusters_v2")
    con.execute("CREATE TABLE developer_clusters_v2 AS SELECT * FROM developer_clusters_v2_df")
    con.unregister("developer_clusters_v2_df")

    print(f"Combined: {len(combined_results):,} developers with cluster labels")
    print("DuckDB table: developer_clusters_v2")
else:
    print("No strata completed.")


Combined: 299,999 developers with cluster labels
DuckDB table: developer_clusters_v2


## 4. Review cluster summary tables

Sorted by size; `noise_flag=1` is the HDBSCAN noise bucket (`hdbscan_cluster == -1`).


In [6]:
for key in STRATA_ORDER:
    path = OUTPUT_DIR / key / "cluster_summary_table.csv"
    if not path.exists():
        print(f"Missing: {path}")
        continue
    print(f"\n### {key} — {STRATA[key]['label']}")
    tbl = pd.read_csv(path)
    cols = [
        "hdbscan_cluster", "n", "pct_of_sample", "noise_flag",
        "behavioral_segment_name", "top_persona", "top_persona_pct",
        "top_journey_state_30d", "top_lifecycle_status", "segment_description",
    ]
    cols = [c for c in cols if c in tbl.columns]
    display(tbl[cols].head(25))



### active — Active (recent 30d activity)


,hdbscan_cluster,n,pct_of_sample,noise_flag,behavioral_segment_name,top_persona,top_persona_pct,top_journey_state_30d,top_lifecycle_status,segment_description
0,52,4311,4.31,0,Explorers,GenAI,86.62,Evaluator,Tourist,"Explorers (active): n=4,311 (4.31% of sample)...."
1,15,3952,3.95,0,Explorers,GenAI,97.57,Evaluator,Active_Discover,"Explorers (active): n=3,952 (3.95% of sample)...."
2,19,3880,3.88,0,Explorers,Unknown,100.00,Evaluator,Tourist,"Explorers (active): n=3,880 (3.88% of sample)...."
3,13,3062,3.06,0,Explorers,CUDA,93.63,Learner,Tourist,"Explorers (active): n=3,062 (3.06% of sample)...."
4,11,3048,3.05,0,Explorers,GenAI,94.52,Evaluator,Active_Discover,"Explorers (active): n=3,048 (3.05% of sample)...."
5,24,2562,2.56,0,Explorers,GenAI,100.00,Evaluator,Tourist,"Explorers (active): n=2,562 (2.56% of sample)...."
6,51,2281,2.28,0,Explorers,GenAI,92.99,Evaluator,Tourist,"Explorers (active): n=2,281 (2.28% of sample)...."
7,25,2226,2.23,0,Explorers,GenAI,100.00,Evaluator,Tourist,"Explorers (active): n=2,226 (2.23% of sample)...."
8,21,2175,2.18,0,Explorers,GenAI,100.00,Evaluator,Tourist,"Explorers (active): n=2,175 (2.18% of sample)...."
9,26,1896,1.90,0,Explorers,GenAI,100.00,Evaluator,Tourist,"Explorers (active): n=1,896 (1.9% of sample). ..."



### cooling — Cooling


,hdbscan_cluster,n,pct_of_sample,noise_flag,behavioral_segment_name,top_persona,top_persona_pct,top_journey_state_30d,top_lifecycle_status,segment_description
0,14,7191,7.19,0,Cooling_Explorers,CUDA,96.13,Cooling_Historically_Active,Tourist,"Cooling_Explorers (cooling): n=7,191 (7.19% of..."
1,19,4185,4.18,0,Cooling_Explorers,Unknown,100.00,Cooling_Historically_Active,Tourist,"Cooling_Explorers (cooling): n=4,185 (4.18% of..."
2,37,3272,3.27,0,Cooling_Explorers,GenAI,86.28,Cooling_Historically_Active,Tourist,"Cooling_Explorers (cooling): n=3,272 (3.27% of..."
3,10,3137,3.14,0,Cooling_Explorers,GenAI,44.34,Cooling_Historically_Active,Tourist,"Cooling_Explorers (cooling): n=3,137 (3.14% of..."
4,35,3122,3.12,0,Cooling_Learners,GenAI,81.55,Cooling_Historically_Active,Tourist,"Cooling_Learners (cooling): n=3,122 (3.12% of ..."
5,3,2690,2.69,0,Cooling_Explorers,GenAI,70.04,Cooling_Historically_Active,Tourist,"Cooling_Explorers (cooling): n=2,690 (2.69% of..."
6,33,2684,2.68,0,Cooling_Explorers,GenAI,100.00,Cooling_Historically_Active,Tourist,"Cooling_Explorers (cooling): n=2,684 (2.68% of..."
7,2,2646,2.65,0,Cooling_Explorers,GenAI,91.99,Cooling_Historically_Active,Tourist,"Cooling_Explorers (cooling): n=2,646 (2.65% of..."
8,11,2486,2.49,0,Cooling_Explorers,GenAI,42.96,Cooling_Historically_Active,Tourist,"Cooling_Explorers (cooling): n=2,486 (2.49% of..."
9,9,2427,2.43,0,Cooling_Explorers,CUDA,49.44,Cooling_Historically_Active,Tourist,"Cooling_Explorers (cooling): n=2,427 (2.43% of..."



### dormant — Dormant


,hdbscan_cluster,n,pct_of_sample,noise_flag,behavioral_segment_name,top_persona,top_persona_pct,top_journey_state_30d,top_lifecycle_status,segment_description
0,38,3359,3.36,0,Dormant_Former_Builders,CUDA,99.82,Dormant_Historically_Active,Dormant_Build,"Dormant_Former_Builders (dormant): n=3,359 (3...."
1,3,3002,3.00,0,Explorers,CUDA,99.23,Dormant_Historically_Active,Tourist,"Explorers (dormant): n=3,002 (3.0% of sample)...."
2,10,2978,2.98,0,Explorers,GenAI,100.00,Dormant_Historically_Active,Tourist,"Explorers (dormant): n=2,978 (2.98% of sample)..."
3,16,2859,2.86,0,Learners,GenAI,100.00,Dormant_Historically_Active,Tourist,"Learners (dormant): n=2,859 (2.86% of sample)...."
4,31,2316,2.32,0,Dormant_Former_Builders,CUDA,100.00,Dormant_Historically_Active,Tourist,"Dormant_Former_Builders (dormant): n=2,316 (2...."
5,35,2266,2.27,0,Dormant_Former_Explorers,CUDA,99.96,Dormant_Historically_Active,Tourist,"Dormant_Former_Explorers (dormant): n=2,266 (2..."
6,32,2223,2.22,0,Dormant_Former_Builders,CUDA,100.00,Dormant_Historically_Active,Tourist,"Dormant_Former_Builders (dormant): n=2,223 (2...."
7,19,2108,2.11,0,Dormant_Former_Builders,Simulation,100.00,Dormant_Historically_Active,Tourist,"Dormant_Former_Builders (dormant): n=2,108 (2...."
8,37,1699,1.70,0,Explorers,CUDA,97.47,Dormant_Historically_Active,Dormant_Evaluate,"Explorers (dormant): n=1,699 (1.7% of sample)...."
9,0,1683,1.68,0,Learners,Learning_Community,71.48,Dormant_Historically_Active,Tourist,"Learners (dormant): n=1,683 (1.68% of sample)...."


## 5. Optional: UMAP visualization (not used for clustering)

Set `RUN_UMAP_VIZ = True` to regenerate embeddings and PNGs (~10–20 min per stratum).


In [7]:
RUN_UMAP_VIZ = False

if RUN_UMAP_VIZ:
    for key in STRATA_ORDER:
        print(f"UMAP viz: {key}...")
        res = run_stratum(con, key, fit_umap=True)
        if res.skipped:
            continue
        r = pd.read_parquet(res.output_dir / "cluster_results.parquet")
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.scatter(r["umap_x"], r["umap_y"], c=r["hdbscan_cluster"], s=3, cmap="tab20", alpha=0.5)
        ax.set_title(f"{STRATA[key]['label']}: HDBSCAN on SVD (noise {res.noise_pct}%)")
        ax.set_xlabel("UMAP-1")
        ax.set_ylabel("UMAP-2")
        plt.tight_layout()
        plt.savefig(res.output_dir / "umap_clusters.png", dpi=150)
        plt.show()
else:
    print("Skipping UMAP viz (RUN_UMAP_VIZ=False).")


Skipping UMAP viz (RUN_UMAP_VIZ=False).


## 6. Next steps (manual review)

1. Open `outputs/clustering/v2/cluster_summary_table_all.csv`.
2. Refine `behavioral_segment_name` per cluster if needed; add `stakeholder_segment_name` in Excel optional.
3. Join labels via `cluster_results_all.parquet` or DuckDB `developer_clusters_v2`.

**Label key:** `(dormancy_segment, hdbscan_cluster)` — IDs are not comparable across strata.
